# Example 11.1 / 5: Quadratic Constraints and Solver Support

Companion notebook for **11.1 Quadratic Models** in the AMPL lab Documentation, section "Quadratic Constraints and Solver Support". It runs the page's example from top to bottom: the model, the amplpy way (data.py, run.py), then the pure-AMPL way (data.dat, model.run). Solver: HiGHS.

## Install Dependencies
💡 TIP: Visit [portal.ampl.com](https://portal.ampl.com/user/ampl/license/list) or [dev.ampl.com](https://dev.ampl.com/ampl/install.html) for AMPL setup on Colab, Docker, Python, Windows, Linux, macOS

In [ ]:
# Install dependencies
%pip install -q amplpy

# Google Colab & Kaggle integration
from amplpy import AMPL, ampl_notebook
ampl = ampl_notebook(
    modules=["highs"],      # modules to install
    license_uuid="default") # license to use

## Quadratic Constraints and Solver Support

A constraint such as

$$\sum_i x_i^2\le R^2$$

defines a convex ball. Reversing it to $\sum_i x_i^2\ge R^2$ generally defines a nonconvex feasible set. Equivalent-looking algebra can also obscure a structure that a driver recognizes, so inspect the generated model and the solver documentation when conic or quadratic recognition matters.

A QCP with the convex ball $x^2+y^2\le1$: minimizing $x+y$ reaches the boundary at
$(-1/\sqrt2,-1/\sqrt2)$ with objective $-\sqrt2\approx-1.41421$. Ipopt, a nonlinear solver,
accepts the quadratic constraint as it is written:

### model.mod
The model, written to a file that both ways of running it read.

In [2]:
%%writefile model.mod
var x;  # first coordinate
var y;  # second coordinate

minimize Sum: x + y;  # linear objective; optimum on the circle

subject to Ball: x^2 + y^2 <= 1;  # convex quadratic constraint: unit disk

Writing model.mod


### amplpy: run.py

`ampl` is the `AMPL` object created above. Each run starts from a fresh session that has read `model.mod`; then `run.py` solves and reads the results.

In [3]:
# A fresh session that has read model.mod, as data.py and run.py assume
ampl.reset()
ampl.eval("reset options;")  # reset keeps the options
ampl.read("model.mod")

In [4]:
# run.py
ampl.solve(solver="ipopt", verbose=False)
print(ampl.solve_result)
print(round(ampl.var["x"].value(), 6), round(ampl.var["y"].value(), 6))
print(round(ampl.obj["Sum"].value(), 6), round(ampl.con["Ball"].body(), 6))

solved
-0.707107 -0.707107
-1.414214 1.0


### Another way in amplpy

A solver that accepts a quadratic objective may still not accept quadratic constraints, so
confirm what the selected driver supports: QP objectives, quadratic constraints, nonconvex
quadratic forms, and integer variables. HiGHS has no native quadratic constraints, so its MP
driver replaces the squares with piecewise-linear approximations and warns about it
(`PLApprox_PowConstExp` in the solver log, which `verbose=False` hides below). With the same
`model.mod`, the objective HiGHS reports, about $-1.41674$, is below the true optimum $-\sqrt2$
because the approximated point slightly violates the original ball (left side about $1.0074$):

In [5]:
# A fresh session that has read model.mod, as data.py and run.py assume
ampl.reset()
ampl.eval("reset options;")  # reset keeps the options
ampl.read("model.mod")

In [6]:
# run.py
ampl.solve(solver="highs", verbose=False)
print(ampl.solve_result)
print(round(ampl.obj["Sum"].value(), 5), round(ampl.con["Ball"].body(), 5))

solved
-1.41674 1.0074
